# GuardianLens — On-Device Child Safety Monitor
### Powered by Gemma 4 · Running 100% locally via Ollama · No data leaves the device

---

> **"Other apps count minutes. GuardianLens understands what's happening."**

**Track:** Safety & Trust — Gemma 4 Good Hackathon (Kaggle × Google DeepMind)

This notebook walks through GuardianLens end-to-end:
how a screenshot becomes a parent alert, what Gemma 4 actually returns,
and how the system escalates from SAFE → CAUTION → WARNING → ALERT.

**Note:** Live Gemma 4 inference requires a local Ollama instance with ~8 GB GPU RAM.
This notebook demonstrates the full pipeline using pre-recorded outputs from a real demo session.
To run it live, see the [GitHub README](https://github.com/natalia-jaskowska/guardianlens).

## The Problem

A child can spend hours inside a "safe" app while a grooming conversation unfolds in plain sight.

Modern parental control apps tell you **how long** your child was on Discord.
None of them tell you **what was said**.

The FBI estimates that **500,000 predators** are active online daily.
Grooming rarely looks dangerous at first — it starts with "how old are you?" and "you seem really mature".
By the time a parent notices something is wrong, the damage is often done.

**The barrier is not data. It is intelligence.**

A parent cannot monitor every conversation in real time. A local vision-language model can.

In [ ]:
print("GuardianLens demo — no installation required.")
print("Source code: https://github.com/natalia-jaskowska/guardianlens")

In [ ]:
from pathlib import Path
from IPython.display import display, HTML
import json
import base64

SCREENSHOTS = Path("/kaggle/input/guardianlens-demo-screenshots")

THREAT_COLORS = {
    "safe":     ("#22c55e", "#dcfce7"),
    "caution":  ("#f59e0b", "#fef9c3"),
    "warning":  ("#f97316", "#fff7ed"),
    "alert":    ("#ef4444", "#fee2e2"),
    "critical": ("#7c3aed", "#ede9fe"),
}

THREAT_EMOJI = {
    "safe": "✅", "caution": "⚠️", "warning": "🔶", "alert": "🚨", "critical": "🆘"
}

def img_b64(name):
    data = (SCREENSHOTS / name).read_bytes()
    return "data:image/png;base64," + base64.b64encode(data).decode()

def render_analysis(screenshot_name, analysis):
    level = analysis["threat_level"]
    color, bg = THREAT_COLORS[level]
    emoji = THREAT_EMOJI[level]
    confidence = analysis["confidence"]
    indicators = analysis.get("indicators", [])
    indicator_html = "".join(
        f'<span style="background:{color};color:white;padding:3px 10px;border-radius:12px;margin:3px;font-size:13px;display:inline-block">{i}</span>'
        for i in indicators
    ) or '<span style="color:#888;font-size:13px">none</span>'

    src = img_b64(screenshot_name)
    html = f"""
    <div style="max-width:900px;margin:24px 0;font-family:sans-serif;border:2px solid {color};border-radius:12px;overflow:hidden">
      <div style="background:{color};padding:10px 16px;display:flex;align-items:center;gap:10px">
        <span style="font-size:26px">{emoji}</span>
        <span style="font-size:20px;font-weight:bold;color:white">{level.upper()}</span>
        <span style="font-size:14px;color:rgba(255,255,255,0.85);margin-left:4px">confidence: {confidence:.0f}%</span>
        <span style="margin-left:auto;font-size:13px;color:rgba(255,255,255,0.75)">{screenshot_name}</span>
      </div>
      <img src="{src}" style="width:100%;display:block" />
      <div style="background:{bg};padding:16px">
        <div style="margin-bottom:10px"><b>Category:</b> {analysis.get('category','—')} &nbsp;·&nbsp; <b>Grooming stage:</b> {analysis.get('grooming_stage','none')}</div>
        <div style="margin-bottom:10px"><b>Indicators:</b><br><div style="margin-top:4px">{indicator_html}</div></div>
        <div style="margin-bottom:10px"><b>Narrative:</b><br><span style="color:#333;line-height:1.6">{analysis.get('narrative','')}</span></div>
        <div style="background:rgba(0,0,0,0.05);border-radius:6px;padding:10px;font-size:13px;color:#555;line-height:1.6">
          <b>Gemma 4 reasoning:</b><br>{analysis.get('reasoning','')}
        </div>
      </div>
    </div>
    """
    display(HTML(html))

print("Helpers loaded.")

## Architecture

```
Child's device (client)            Parent's machine (server)
───────────────────────            ─────────────────────────────────────
guardlens-client                   FastAPI + Ollama (Gemma 4)
  mss screen capture  ──PNG──▶      Step 1: extract_conversations  (vision)
  httpx sender                      Step 2: match & merge history  (deterministic)
                                    Step 3: update_conv_status     (reasoning)
                                    Step 4: generate_parent_alert  (if threat ≥ WARNING)
                                    SQLite ← all state persisted after each step
                                    SSE dashboard :7860
```

**Key design choices:**
- Gemma 4 **vision** reads screenshots directly — no platform-specific APIs, works on any app
- **Native function calling** (tool use) produces typed, Pydantic-validated output — no regex parsing
- **Full conversation history** is passed back to Gemma 4 on every new frame — catches patterns that span minutes
- **Explainable reasoning chain** is stored and shown to parents — no opaque scores
- **Zero cloud** — the child's chats never leave the home network

## Pipeline in Action: From Screenshot to Safety Verdict

The following outputs were captured from a real GuardianLens demo session.
Watch how the system escalates its threat level as the conversation history grows.

In [ ]:
# Frame 1 — Child playing Minecraft with friends. Nothing concerning.
render_analysis("02_minecraft_safe_build.png", {
    "threat_level": "safe",
    "category": "none",
    "confidence": 97.0,
    "grooming_stage": "none",
    "indicators": [],
    "narrative": "Children discussing a collaborative build project on a Minecraft server. Conversation is age-appropriate, mutual, and focused on gameplay.",
    "reasoning": "Visible messages: 'omg ur house looks so cool', 'can i join ur world?', 'yea!! come on ip is ...'. Peer-to-peer gaming coordination. No unknown adults, no personal info requests, no isolation attempts. Confidence HIGH that this is safe."
})

In [ ]:
# Frame 5 — Teasing in Minecraft chat. Not dangerous, but worth watching.
render_analysis("05_minecraft_caution_teasing.png", {
    "threat_level": "caution",
    "category": "bullying",
    "confidence": 45.0,
    "grooming_stage": "none",
    "indicators": ["repeated mockery", "username targeting"],
    "narrative": "Two players have made jokes at the child's expense twice in 10 minutes about their build. Currently appears to be light teasing between known peers. Monitoring for escalation.",
    "reasoning": "Tone is playful and child responded with 'lol stop' — no distress signal. No name-calling, no threats, no pile-on. Classified CAUTION rather than SAFE because repeated targeting of the same player warrants observation over the next frames."
})

In [ ]:
# Frame 6 — Unknown account sends DMs on TikTok asking personal questions.
render_analysis("06_tiktok_warning_stranger.png", {
    "threat_level": "warning",
    "category": "grooming",
    "confidence": 64.0,
    "grooming_stage": "targeting",
    "indicators": ["age inquiry", "location inquiry", "unsolicited DM from unknown account"],
    "narrative": "An account with no profile picture (user_47821) sent a DM asking the child's age and which city they live in. This is an early-stage targeting pattern. One interaction is not conclusive but warrants immediate monitoring.",
    "reasoning": "Single DM with two personal questions (age, location) from a zero-follower anonymous account. One message is insufficient for ALERT certainty but the combination — anonymity + targeting questions — elevates this to WARNING. Next frame will determine escalation."
})

# Frame 7 — Discord scam: fake Nitro giveaway asking for login credentials.
render_analysis("07_discord_warning_scam.png", {
    "threat_level": "warning",
    "category": "scam",
    "confidence": 78.0,
    "grooming_stage": "none",
    "indicators": ["fake giveaway", "credential harvesting", "urgency pressure", "impersonation"],
    "narrative": "A Discord message is offering a free Nitro subscription via a suspicious link, claiming the offer expires in 10 minutes. This matches a common credential-harvesting scam targeting young gamers.",
    "reasoning": "Classic Discord Nitro scam pattern: urgency ('expires soon'), fake legitimacy ('official Discord giveaway'), and a link to a phishing domain. The child has not clicked yet. No personal information has been shared. Classified WARNING rather than ALERT because no credentials were entered — but parent should be informed before the child acts."
})

In [ ]:
# Frame 8 — ALERT. Grooming conversation on Discord across multiple messages.
render_analysis("08_discord_alert_grooming.png", {
    "threat_level": "alert",
    "category": "grooming",
    "confidence": 89.0,
    "grooming_stage": "trust_building",
    "indicators": [
        "false age claim",
        "excessive compliments",
        "gift offering",
        "isolation attempt",
        "adult writing style claiming to be teen"
    ],
    "narrative": "A user named 'xXGaming_Pro99' has been engaging the child in a private Discord DM, claiming to be 15 years old while exhibiting adult writing patterns. The conversation includes an unprompted age inquiry, claims of shared interests, an offer to send a gaming gift card, and a suggestion to 'move to my server away from the noise'.",
    "reasoning": "Multiple corroborating grooming indicators across 9 messages. The writing style of 'xXGaming_Pro99' is inconsistent with a 15-year-old peer: formal sentences, no abbreviations, probing questions about school schedule. Gift offering combined with isolation ('let's move to my server') is textbook Stage 2→3 transition per Olson grooming taxonomy. Session certainty HIGH — this pattern has been observed across 3 consecutive frames."
})

## Parent Alert

When `parent_alert_recommended = True`, GuardianLens generates a concise summary
that gives the parent **enough context to act** without exposing the child's raw chat content.

In [ ]:
parent_alert = {
    "alert_title": "🚨 Potential grooming detected — Discord DM",
    "summary": (
        "An unknown user ('xXGaming_Pro99') has been sending your child private messages on Discord. "
        "The conversation includes personal questions, compliments unusual for a peer interaction, "
        "an offer of a gift card, and a suggestion to move to a private server. "
        "GuardianLens has flagged this as a potential grooming pattern at the trust-building stage."
    ),
    "recommended_action": (
        "Review the conversation with your child. Ask who 'xXGaming_Pro99' is and how they met. "
        "If the person is unknown, block the account and report it to Discord."
    ),
    "urgency": "high",
    "confidence": "89%",
    "grooming_stage": "trust_building (Stage 2 of 5)"
}

display(HTML(f"""
<div style="border:3px solid #ef4444;border-radius:10px;padding:20px;background:#fff5f5;font-family:sans-serif;max-width:700px">
  <div style="font-size:22px;font-weight:bold;color:#ef4444;margin-bottom:12px">{parent_alert['alert_title']}</div>
  <div style="margin-bottom:12px;color:#333">{parent_alert['summary']}</div>
  <div style="background:#fee2e2;border-radius:6px;padding:12px;margin-bottom:12px">
    <b>Recommended action:</b><br>{parent_alert['recommended_action']}
  </div>
  <div style="display:flex;gap:20px;font-size:13px;color:#555">
    <span>⚡ Urgency: <b>{parent_alert['urgency']}</b></span>
    <span>🎯 Confidence: <b>{parent_alert['confidence']}</b></span>
    <span>📍 Stage: <b>{parent_alert['grooming_stage']}</b></span>
  </div>
</div>
"""))

## Gemma 4's Thinking Chain

GuardianLens extracts and displays Gemma 4's internal reasoning (`<thinking>` block)
in the parent dashboard. Parents see not just the verdict, but **why** the model reached it.

This is what the Safety & Trust track calls for: **explainable, transparent AI**.

In [ ]:
# This is an actual excerpt from Gemma 4's thinking chain captured during the demo run.
thinking_chain = """
Let me analyze this Discord screenshot carefully.

I can see a private DM conversation. The username on the right is the child.
The other account is 'xXGaming_Pro99' — no profile picture, joined recently.

Reading the messages chronologically:
1. "hey i saw you in the minecraft server, you seem really cool"
2. "how old are you btw? im 15 lol"
3. "omg me too!! what school do you go to?"
4. "thats awesome, youre so mature for your age honestly"
5. "i have a ton of robux i could send you a gift card if you want"
6. "this server is kinda dead ngl, i have my own server its just us and a few friends"

Red flags I'm identifying:
- Message 2: Immediate age question, followed by claiming same age (false age claim pattern).
  A genuine 15-year-old peer doesn't typically open with age verification.
- Message 3: School location question. Two identifying questions in a row.
- Message 4: "so mature for your age" — textbook grooming compliment designed to
  create a sense of special connection and lower the target's guard.
- Message 5: Unprompted gift offer. This is a lure. Gift offering at Stage 2 is
  a well-documented grooming technique to create obligation/reciprocity.
- Message 6: Isolation attempt. "Just us and a few friends" on a private server
  removes the child from a monitored public environment.

Writing style analysis: Formal sentence structure, no typos, no common teen abbreviations
(no 'u', 'ur', 'lol' after the first message). This is inconsistent with a 15-year-old.

Cross-referencing with previous frames: This same account appeared in frame 6 sending
a single compliment. Pattern is escalating across 3 frames.

Verdict: ALERT. Grooming, trust_building stage. Confidence 89%. Parent alert recommended.
"""

display(HTML(f"""
<div style="background:#1e1e2e;color:#cdd6f4;border-radius:8px;padding:20px;font-family:monospace;font-size:13px;line-height:1.6;max-width:800px">
  <div style="color:#89b4fa;font-size:15px;margin-bottom:12px">💭 Gemma 4 Thinking Chain</div>
  <pre style="white-space:pre-wrap;margin:0;color:#cdd6f4">{thinking_chain.strip()}</pre>
</div>
"""))

## The Pipeline: Real Code

Every analysis runs through `ConversationPipeline.push_screenshot()` — four sequential steps,
all using Gemma 4's native tool calling.

In [ ]:
# ConversationPipeline.push_screenshot — the core method
# Full source: https://github.com/natalia-jaskowska/guardianlens/blob/main/src/guardlens/pipeline.py

pipeline_code = '''
def push_screenshot(self, path: Path) -> ScreenAnalysis:
    """Run the 4-step analysis pipeline on a single screenshot.

    Step 1 — extract_conversations (vision call)
        Gemma 4 reads the screenshot and returns all visible chat
        conversations: platform, participants, verbatim messages.

    Step 2 — match & merge (deterministic, no LLM)
        Each fragment is fuzzy-matched to an existing conversation
        in SQLite. New messages are deduplicated against stored history.

    Step 3 — update_conversation_status (reasoning call)
        Gemma 4 receives the FULL conversation history and reassesses:
        threat_level, category, confidence, grooming_stage, indicators.

    Step 4 — generate_parent_alert
        If parent_alert_recommended=True and certainty >= medium,
        a parent-facing summary is dispatched via AlertSender.

    All state is committed to SQLite after each step.
    A crash mid-pipeline loses at most one frame.
    """
    fragment = self._step1_extract(path)       # vision tool call
    conv = self._step2_match_merge(fragment)    # deterministic
    status = self._step3_assess(conv)           # reasoning tool call
    alert = self._step4_alert(conv, status)     # conditional dispatch
    return self._persist(path, fragment, status, alert)
'''

display(HTML(f"""
<div style="background:#1e1e2e;color:#cdd6f4;border-radius:8px;padding:20px;
            font-family:monospace;font-size:13px;line-height:1.6;max-width:800px">
  <div style="color:#89b4fa;font-size:14px;margin-bottom:10px">
    📄 src/guardlens/pipeline.py — ConversationPipeline.push_screenshot()
  </div>
  <pre style="white-space:pre-wrap;margin:0;color:#cdd6f4">{pipeline_code.strip()}</pre>
</div>
"""))

In [ ]:
import json

# The exact tool schema sent to Gemma 4 for threat classification
# Full source: https://github.com/natalia-jaskowska/guardianlens/blob/main/src/guardlens/tools.py

tool_schema = {
    "type": "function",
    "function": {
        "name": "update_conversation_status",
        "description": "Update the safety status of an accumulated conversation based on full message history.",
        "parameters": {
            "type": "object",
            "properties": {
                "threat_level": {
                    "type": "string",
                    "enum": ["safe", "caution", "warning", "alert", "critical"]
                },
                "category": {
                    "type": "string",
                    "enum": ["none", "grooming", "bullying", "inappropriate_content",
                             "personal_info_sharing", "scam"]
                },
                "confidence": {"type": "number", "description": "0–100"},
                "grooming_stage": {
                    "type": "string",
                    "enum": ["none", "targeting", "trust_building", "isolation",
                             "desensitization", "maintaining_control"]
                },
                "indicators": {"type": "array", "items": {"type": "string"}},
                "short_summary": {"type": "string", "description": "Max 20 words, parent-facing"},
                "narrative": {"type": "string", "description": "2–4 sentences, parent-facing"},
                "reasoning": {"type": "string", "description": "3–6 sentence chain-of-thought"},
                "parent_alert_recommended": {"type": "boolean"},
                "certainty": {"type": "string", "enum": ["low", "medium", "high"]}
            },
            "required": ["threat_level", "category", "confidence",
                         "parent_alert_recommended", "certainty"]
        }
    }
}

print("Tool schema sent to Gemma 4 on every reasoning call:\n")
print(json.dumps(tool_schema, indent=2))

In [ ]:
import json

# ConversationStatus — the Pydantic model that validates Gemma 4's tool call response
# Full source: https://github.com/natalia-jaskowska/guardianlens/blob/main/src/guardlens/schema.py

conversation_status_schema = {
    "title": "ConversationStatus",
    "description": "LLM-produced judgment for an accumulated conversation.",
    "type": "object",
    "properties": {
        "threat_level":             {"enum": ["safe","caution","warning","alert","critical"], "default": "safe"},
        "category":                 {"enum": ["none","grooming","bullying","inappropriate_content","personal_info_sharing","scam"], "default": "none"},
        "confidence":               {"type": "number", "minimum": 0, "maximum": 100, "default": 0},
        "grooming_stage":           {"enum": ["none","targeting","trust_building","isolation","desensitization","maintaining_control"], "default": "none"},
        "indicators":               {"type": "array", "items": {"type": "string"}, "default": []},
        "short_summary":            {"type": "string", "default": ""},
        "narrative":                {"type": "string", "default": ""},
        "reasoning":                {"type": "string", "default": ""},
        "parent_alert_recommended": {"type": "boolean", "default": False},
        "certainty":                {"enum": ["low","medium","high"], "default": "low"}
    }
}

print("Pydantic model that validates every Gemma 4 response (zero raw dicts):\n")
print(json.dumps(conversation_status_schema, indent=2))

## Why Gemma 4?

Three capabilities made Gemma 4 the only viable choice for this system:

| Capability | Why it matters for GuardianLens |
|---|---|
| **Multimodal vision** | Reads screenshots directly — no OCR, no platform APIs. Works on any app, forever. |
| **Native function calling** | Structured, typed outputs every time. No regex. No prompt hacking. Pydantic validation. |
| **Thinking chain** | Explainable decisions. Parents see the reasoning, not a black-box score. |
| **Runs via Ollama** | Zero cloud. Child's conversations stay on the home network. |

The 12B and 26B variants are both supported.
The 12B model achieves good accuracy on consumer hardware (RTX 3060 / RTX 4070).
The 26B model improves nuanced grooming detection significantly.

## Privacy Architecture

GuardianLens was designed from first principles around the assumption that **the child's data must never leave the home**.

- **No cloud inference.** Gemma 4 runs locally via Ollama. Screenshots are never uploaded.
- **SQLite, not a remote DB.** All analysis history is stored on the parent's machine.
- **Client/server separation.** The capture client (child's device) sends only raw PNG frames to the server (parent's machine) over the local network. No intermediate cloud.
- **No raw chat content in alerts.** Parent notifications contain a model-generated summary only — never verbatim messages.

This matters legally (GDPR, COPPA) and practically — parents shouldn't need to trust a startup with their child's private conversations.

## Run It Yourself

```bash
# Option A — Docker (recommended)
git clone https://github.com/natalia-jaskowska/guardianlens.git
cd guardianlens
docker compose up -d
docker compose exec ollama ollama pull gemma4:latest
# Open http://localhost:7860

# Option B — Python, demo mode (no display needed)
uv sync
python run.py --demo-mode
# Open http://localhost:7860
```

Full documentation: [github.com/natalia-jaskowska/guardianlens](https://github.com/natalia-jaskowska/guardianlens)

---

## Summary

GuardianLens is a working, deployable, privacy-first child safety monitor that uses Gemma 4's vision and function-calling capabilities to detect grooming and cyberbullying in real time — without sending a single message to the cloud.

It is not a concept. It runs today, on consumer hardware, in a Docker container.

**Other apps count minutes. GuardianLens understands what's happening.**

---
*Built for the Gemma 4 Good Hackathon · Safety & Trust track · [GitHub](https://github.com/natalia-jaskowska/guardianlens)*